In [ ]:
import os
import pandas as pd
import numpy as np
import requests
import yfinance as yf
import wbdata
from pycoingecko import CoinGeckoAPI
from datetime import datetime

os.makedirs("real_data", exist_ok=True)

# ============================================================
# 1. SHILLER S&P 500 (Yale University)
# ============================================================

print("Downloading Shiller S&P 500 (Yale)...")

shiller_url = "https://www.econ.yale.edu/~shiller/data/ie_data.xls"
shiller = pd.read_excel(shiller_url, sheet_name="Data", skiprows=7)

shiller = shiller.rename(columns={
    "Date": "Date",
    "P": "Price",
    "CAPE": "CAPE",
    "E10": "PE10"
})

shiller = shiller[["Date", "Price", "CAPE", "PE10"]].dropna()
shiller["Date"] = pd.to_datetime(shiller["Date"])

shiller.to_csv("real_data/shiller_sp500.csv", index=False)
print("✓ Shiller dataset saved.")


# ============================================================
# 2. FRED (Federal Reserve Economic Data)
# ============================================================

print("Downloading FRED interest rates...")

fred_series = {
    "DGS10": "10y_treasury",
    "DTB3": "3m_tbill",
    "DGS30": "30y_treasury",
    "FEDFUNDS": "fed_funds",
    "T10Y3M": "term_spread_10y3m",
    "T10Y2Y": "term_spread_10y2y",
    "VIXCLS": "vix"
}

fred_data = {}

for code, name in fred_series.items():
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
    df = pd.read_csv(url)
    df.columns = ["DATE", name]
    df["DATE"] = pd.to_datetime(df["DATE"])
    fred_data[name] = df

fred_merged = fred_data["10y_treasury"]
for name, df in fred_data.items():
    fred_merged = fred_merged.merge(df, on="DATE", how="outer")

fred_merged.to_csv("real_data/fred_rates.csv", index=False)
print("✓ FRED dataset saved.")


# ============================================================
# 3. GLOBAL EQUITY INDICES (Yahoo Finance)
# ============================================================

print("Downloading global equity indices...")

tickers = {
    "SP500": "^GSPC",
    "FTSE100": "^FTSE",
    "DAX": "^GDAXI",
    "NIKKEI": "^N225",
    "TSX": "^GSPTSE",
    "KOSPI": "^KS11",
    "CAC40": "^FCHI",
    "AEX": "^AEX",
    "SMI": "^SSMI",
    "IBEX": "^IBEX"
}

eq = yf.download(list(tickers.values()), start="1980-01-01",
                 interval="1mo", auto_adjust=True, progress=False)["Close"]

eq.columns = list(tickers.keys())
eq.to_csv("real_data/global_equities.csv")
print("✓ Global equity indices saved.")


# ============================================================
# 4. HOUSING DATA (BIS — Bank for International Settlements)
# ============================================================

print("Downloading BIS housing prices...")

bis_url = "https://www.bis.org/statistics/full_data_sets/pp_detailed_csv.zip"
bis_zip = requests.get(bis_url)

with open("real_data/bis_house_prices.zip", "wb") as f:
    f.write(bis_zip.content)

print("✓ BIS housing ZIP downloaded (extract manually).")


# ============================================================
# 5. WORLD BANK MACRO PANEL
# ============================================================

print("Downloading World Bank macro indicators...")

wb_indicators = {
    "NY.GDP.MKTP.KD.ZG": "gdp_growth",
    "FP.CPI.TOTL.ZG": "inflation",
    "FR.INR.LEND": "lending_rate",
    "GC.DOD.TOTL.GD.ZS": "govt_debt_pct_gdp",
    "FS.AST.DOMO.GD.ZS": "domestic_credit_pct_gdp",
    "SL.UEM.TOTL.ZS": "unemployment"
}

countries = ["US","GB","DE","FR","JP","CA","AU","KR","CN","IN"]

wb = wbdata.get_dataframe(wb_indicators, country=countries)
wb = wb.reset_index()
wb.to_csv("real_data/worldbank_macro.csv", index=False)

print("✓ World Bank macro panel saved.")


# ============================================================
# 6. OIL & GOLD (Real Institutional Sources)
# ============================================================

print("Downloading oil & gold prices...")

# Brent Oil (EIA)
oil_url = "https://www.eia.gov/dnav/pet/hist_xls/RBRTEd.xls"
oil = pd.read_excel(oil_url, skiprows=2)
oil.columns = ["Date", "Brent"]
oil.to_csv("real_data/brent_oil.csv", index=False)

# Gold (LBMA)
gold_url = "https://datahub.io/core/gold-prices/r/monthly.csv"
gold = pd.read_csv(gold_url)
gold.to_csv("real_data/gold_prices.csv", index=False)

print("✓ Oil & gold saved.")


# ============================================================
# 7. CRYPTO (CoinGecko + CryptoDataDownload)
# ============================================================

print("Downloading crypto data...")

cg = CoinGeckoAPI()

btc = cg.get_coin_market_chart_by_id("bitcoin", "usd", 365)
eth = cg.get_coin_market_chart_by_id("ethereum", "usd", 365)

btc_df = pd.DataFrame(btc["prices"], columns=["ts", "price"])
btc_df["date"] = pd.to_datetime(btc_df["ts"], unit="ms")
btc_df.to_csv("real_data/btc_365d.csv", index=False)

eth_df = pd.DataFrame(eth["prices"], columns=["ts", "price"])
eth_df["date"] = pd.to_datetime(eth_df["ts"], unit="ms")
eth_df.to_csv("real_data/eth_365d.csv", index=False)

# Full history (non‑Kaggle)
btc_full = pd.read_csv("https://www.cryptodatadownload.com/cdd/Binance_BTCUSDT_d.csv", skiprows=1)
btc_full.to_csv("real_data/btc_full.csv", index=False)

eth_full = pd.read_csv("https://www.cryptodatadownload.com/cdd/Binance_ETHUSDT_d.csv", skiprows=1)
eth_full.to_csv("real_data/eth_full.csv", index=False)

print("✓ Crypto datasets saved.")


# ============================================================
# 8. IMF CRISIS DATES (Laeven–Valencia)
# ============================================================

print("Downloading IMF crisis dates...")

lv_url = "https://www.imf.org/-/media/Files/Publications/WP/2020/English/wpiea2020145-data.ashx"
lv = requests.get(lv_url)

with open("real_data/laeven_valencia_2020.xlsx", "wb") as f:
    f.write(lv.content)

print("✓ Laeven–Valencia crisis dataset saved.")
